In [5]:
# last
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    get_response_synthesizer,
    StorageContext,
    load_index_from_storage,
)
from llama_parse import LlamaParse
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    CorrectnessEvaluator,
)
import pandas as pd
import asyncio
import json
import time
import nest_asyncio
import os
from types import SimpleNamespace
from llama_index.core.llama_dataset import (
    LabelledRagDataExample,
)
import itertools

# Apply nested asyncio for notebook execution
nest_asyncio.apply()


class RAGEvaluator:
    def __init__(
        self,
        data_directory="data",
        dataset_file="./rag_dataset.json",
        index_persist_dir="./indexes",
    ):
        # Configure settings
        self._configure_settings()

        # Initialize parser and file extractor
        self.parser = LlamaParse(result_type="markdown")
        self.file_extractor = {".pdf": self.parser}

        # Set data directory and persist directory
        self.data_directory = data_directory
        self.index_persist_dir = index_persist_dir

        # Ensure index directory exists
        os.makedirs(self.index_persist_dir, exist_ok=True)

        # Load dataset
        self.rag_dataset = self._load_dataset(dataset_file)

        # Initialize evaluators
        self.relevancy_evaluator = RelevancyEvaluator()
        self.correctness_evaluator = CorrectnessEvaluator()

        # Initialize results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Initialize summary results DataFrame for different parameter combinations
        self.summary_results_df = pd.DataFrame()

    def _configure_settings(self):
        """Configure global settings for embedding model and LLM"""
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-zh-v1.5")
        Settings.llm = Ollama(
            model="llama3.1:latest", request_timeout=60.0, temperature=0.3
        )

    def _load_documents(self, data_directory):
        """Load documents from the specified directory"""
        return SimpleDirectoryReader(
            data_directory, file_extractor=self.file_extractor
        ).load_data()

    def _load_dataset(self, dataset_file):
        """Load the RAG dataset from a JSON file"""
        with open(dataset_file, "r") as f:
            # Convert dictionary to object with attribute access
            return json.load(f, object_hook=lambda d: SimpleNamespace(**d))

    def _get_index_persist_path(self, chunk_size, chunk_overlap):
        """Get the directory path for persisting the index"""
        # 使用人類可讀的格式
        index_dir = f"chunk_size{chunk_size}_overlap{chunk_overlap}"
        return os.path.join(self.index_persist_dir, index_dir)

    def _create_or_load_index(self, chunk_size, chunk_overlap):
        """Create a new index or load an existing one based on chunk parameters"""
        index_persist_path = self._get_index_persist_path(chunk_size, chunk_overlap)

        if os.path.exists(index_persist_path):
            print(
                f"Loading existing index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            # Load existing index
            storage_context = StorageContext.from_defaults(
                persist_dir=index_persist_path
            )
            index = load_index_from_storage(storage_context)
        else:
            print(
                f"Creating new index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            documents = self._load_documents(self.data_directory)
            # Create text splitter with the specified chunk size and overlap
            text_splitter = SentenceSplitter(
                chunk_size=chunk_size, chunk_overlap=chunk_overlap
            )

            # Create vector index
            index = VectorStoreIndex.from_documents(
                documents, transformations=[text_splitter]
            )

            # Persist the index
            index.storage_context.persist(persist_dir=index_persist_path)

        total_num_chunks = len(index.docstore.docs)
        print(f"Loaded index with {total_num_chunks} documents")

        return index

    def _create_query_engine(self, chunk_size, chunk_overlap, top_k):
        """Create a query engine with the specified parameters"""
        # Create or load index
        index = self._create_or_load_index(chunk_size, chunk_overlap)

        # Configure retrievers with the specified top_k
        vector_retriever = index.as_retriever(similarity_top_k=top_k, verbose=True)
        bm25_retriever = BM25Retriever.from_defaults(
            docstore=index.docstore, similarity_top_k=top_k
        )

        # Create fusion retriever
        retriever = QueryFusionRetriever(
            [vector_retriever, bm25_retriever],
            similarity_top_k=top_k,
            num_queries=1,  # set to 1 to disable query generation
            mode="reciprocal_rerank",
            use_async=True,
            verbose=True,
        )

        # Create response synthesizer
        response_synthesizer = get_response_synthesizer()

        # Create and return query engine
        return RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=response_synthesizer,
        )

    def _add_eval_row(
        self,
        response,
        dataset_example,
        relevancy_result,
        correctness_result,
        response_time,
    ):
        """Add evaluation results to the DataFrame"""
        if not response.source_nodes:
            print("No response!")
            return

        eval_row = pd.DataFrame(
            [
                {
                    "Query": dataset_example.query,
                    "Response": str(response),
                    "Reference Answer": dataset_example.reference_answer,
                    "Source": response.source_nodes[0].node.text,
                    "Relevancy Eval Result": f"{'Pass' if relevancy_result.passing else 'Fail'} \nscore: {relevancy_result.score}",
                    "Relevancy Reasoning": relevancy_result.feedback,
                    "Correctness Eval Result": f"{'Pass' if correctness_result.passing else 'Fail'} \n\nscore: {correctness_result.score}",
                    "Correctness Reasoning": correctness_result.feedback,
                    "Response Time": f"{response_time:.4f}",
                }
            ]
        )

        self.eval_results_df = pd.concat(
            [self.eval_results_df, eval_row], ignore_index=True
        )

    async def _evaluate_engine(
        self, query_engine, dataset_examples: LabelledRagDataExample
    ):
        """Evaluate the query engine on the given examples"""
        # # Limit to first 3 examples for testing
        dataset_examples = dataset_examples[15:17]

        # Initialize tracking variables
        relevancy_total_correct = 0
        correctness_total_correct = 0
        correctness_total_score = 0
        total_response_time = 0

        # Process each example
        for dataset_example in dataset_examples:
            start_time = time.time()
            response = query_engine.query(dataset_example.query)
            response_time = time.time() - start_time

            # Evaluate relevancy and correctness
            relevancy_result = self.relevancy_evaluator.evaluate_response(
                query=dataset_example.query, response=response
            )
            correctness_result = self.correctness_evaluator.evaluate_response(
                query=dataset_example.query,
                response=response,
                reference=dataset_example.reference_answer,
            )

            # Add results to DataFrame
            self._add_eval_row(
                response,
                dataset_example,
                relevancy_result,
                correctness_result,
                response_time,
            )

            # Update totals
            total_response_time += response_time
            if relevancy_result.passing:
                relevancy_total_correct += 1
            if correctness_result.passing:
                correctness_total_correct += 1
                correctness_total_score += correctness_result.score

        return (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            len(dataset_examples),
            total_response_time,
        )

    def evaluate_with_params(self, chunk_size, chunk_overlap, top_k):
        """Run evaluation with the specified parameters"""
        print(
            f"Parameters: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}, top_k={top_k}"
        )

        # Reset results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Create query engine with the specified parameters
        query_engine = self._create_query_engine(chunk_size, chunk_overlap, top_k)

        # Run evaluation
        (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            total_questions,
            total_response_time,
        ) = asyncio.run(self._evaluate_engine(query_engine, self.rag_dataset.examples))

        # Display results
        styled_df = self.eval_results_df.style.set_properties(
            **{"white-space": "pre-wrap"},
        )

        display(styled_df)

        # Calculate scores
        relevancy_score = relevancy_total_correct / total_questions
        correctness_score = correctness_total_score / total_questions
        avg_response_time = total_response_time / total_questions

        # Display summary
        print(
            f"Total Relevancy correct: {relevancy_total_correct} out of {total_questions}, "
            f"score: {relevancy_score}"
        )
        print(
            f"Total Correctness correct: {correctness_total_correct} out of {total_questions}, "
            f"score: {correctness_score}"
        )
        print(f"Average response time: {avg_response_time:.4f} seconds")
        print("===============================================")

        # Add result to summary DataFrame
        self._add_summary_row(
            chunk_size,
            chunk_overlap,
            top_k,
            relevancy_score,
            correctness_score,
            avg_response_time,
        )

        return {
            "relevancy_score": relevancy_score,
            "correctness_score": correctness_score,
            "avg_response_time": avg_response_time,
        }

    def _add_summary_row(
        self,
        chunk_size,
        chunk_overlap,
        top_k,
        relevancy_score,
        correctness_score,
        avg_response_time,
    ):
        """Add a summary row to the summary results DataFrame"""
        summary_row = pd.DataFrame(
            [
                {
                    "Chunk Size": chunk_size,
                    "Chunk Overlap": chunk_overlap,
                    "Top K": top_k,
                    "Relevancy Score": f"{relevancy_score:.4f}",
                    "Correctness Score": f"{correctness_score:.4f}",
                    "Avg Response Time": f"{avg_response_time:.4f}",
                }
            ]
        )

        self.summary_results_df = pd.concat(
            [self.summary_results_df, summary_row], ignore_index=True
        )
        self.summary_results_df.to_csv(
            "summary_results_second_half_last_2.csv", index=False
        )

    def run_evaluations(self, chunk_sizes, overlaps, top_ks):
        """Run evaluations for multiple parameter combinations"""
        # Reset summary results DataFrame
        # self.summary_results_df = pd.DataFrame()

        # Generate all parameter combinations
        param_combinations = list(itertools.product(chunk_sizes, overlaps, top_ks))
        total_combinations = len(param_combinations)

        print(f"Running evaluations for {total_combinations} parameter combinations...")

        results = {}
        for i, (chunk_size, overlap, top_k) in enumerate(param_combinations):
            print(f"\nEvaluation {i+1}/{total_combinations}")
            # Convert overlap from percentage to absolute value
            chunk_overlap = int(chunk_size * overlap)
            results[(chunk_size, overlap, top_k)] = self.evaluate_with_params(
                chunk_size, chunk_overlap, top_k
            )

        # Display summary table
        self._display_summary_table()

        return results

    def _display_summary_table(self):
        """Display a summary table of all parameter combinations"""
        print("\n--- Summary of All Parameter Combinations ---")

        # Sort the summary results by scores
        sorted_df = self.summary_results_df.sort_values(
            by=["Relevancy Score", "Correctness Score"], ascending=False
        )

        display(sorted_df)

        # Find the best parameter combination
        best_row = sorted_df.iloc[0]
        print(f"\nBest Parameter Combination:")
        print(f"Chunk Size: {best_row['Chunk Size']}")
        print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
        print(f"Top K: {best_row['Top K']}")
        print(f"Relevancy Score: {best_row['Relevancy Score']}")
        print(f"Correctness Score: {best_row['Correctness Score']}")
        print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")

In [6]:
evaluator = RAGEvaluator()

# Define parameter ranges to test
# (512,51,8)~
# (512,25,3)~(512,102,12)
top_ks = [7]


chunk_sizes = [192]
overlaps = [0.15]
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

Running evaluations for 1 parameter combinations...

Evaluation 1/1
Parameters: chunk_size=192, chunk_overlap=28, top_k=7
Loading existing index for chunk_size=192, chunk_overlap=28
Loaded index with 112 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,未來大學停車場的收費標準是什麼？,每小時20元，全天封頂150元。,未來大學停車場的收費標準是每小時20元，全天封頂150元。,# 校園內設有多個停車場 - 分佈在主要教學樓和宿舍區附近 - 停車費用: 每小時20元，全天封頂150元 # 電動車充電站 - 多處停車場設有電動車充電樁，支持多種充電標準 # 摩托車 - 摩托車停車區 #,Pass score: 1.0,"YES. The response for the query ""未來大學停車場的收費標準是什麼？"" is in line with the context information provided, which states that ""停車費用: 每小時20元，全天封頂150元"".",Pass score: 4.0,"The generated answer is fully correct and relevant to the user query, with the same metrics as the reference answer. The only difference is that the generated answer lacks a subject (""未來大學停車場""), but this does not affect its correctness or relevance.",3.0608
1,電動車充電站的設施分布在哪些地方？,多處停車場。,停車場。,# 校園內設有多個停車場 - 分佈在主要教學樓和宿舍區附近 - 停車費用: 每小時20元，全天封頂150元 # 電動車充電站 - 多處停車場設有電動車充電樁，支持多種充電標準 # 摩托車 - 摩托車停車區 #,Pass score: 1.0,"YES. The response ""多處停車場"" (multiple parking lots) is in line with the context information provided, which mentions that there are multiple parking lots distributed around the campus and also near electric car charging stations.",Pass score: 4.0,"The generated answer is relevant to the user query and provides a similar level of detail as the reference answer, but it contains some minor differences in wording (""多處"" vs ""停車場""). However, the overall meaning and intent are preserved, making it a correct and relevant response.",0.6558


Total Relevancy correct: 2 out of 2, score: 1.0
Total Correctness correct: 2 out of 2, score: 4.0
Average response time: 1.8583 seconds

--- Summary of All Parameter Combinations ---


,Chunk Size,Chunk Overlap,Top K,Relevancy Score,Correctness Score,Avg Response Time
0,192,28,7,1.0000,4.0000,1.8583



Best Parameter Combination:
Chunk Size: 192
Chunk Overlap: 28
Top K: 7
Relevancy Score: 1.0000
Correctness Score: 4.0000
Avg Response Time: 1.8583 seconds
